In [1]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

Project root: c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence
Raw data directory: c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\data\raw
Processed data directory: c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\data\processed


In [10]:
train_path = RAW_DATA_DIR / "banking77_train.csv"
test_path = RAW_DATA_DIR / "banking77_test.csv"

train_url = (
    "https://raw.githubusercontent.com/PolyAI-LDN/"
    "task-specific-datasets/master/banking_data/train.csv"
)

test_url = (
    "https://raw.githubusercontent.com/PolyAI-LDN/"
    "task-specific-datasets/master/banking_data/test.csv"
)

if train_path.exists() and test_path.exists():
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print("Loaded existing local CSV files.")
else:
    train_df = pd.read_csv(train_url)
    test_df = pd.read_csv(test_url)

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Downloaded and saved the CSV files.")

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

Downloaded and saved the CSV files.
Training shape: (10003, 2)
Test shape: (3080, 2)


In [11]:
print("\nTraining columns:", train_df.columns.tolist())
print("\nFirst five training rows:")
display(train_df.head())


Training columns: ['text', 'category']

First five training rows:


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


### Inspect the dataset structure

In [12]:
print("Training dataset information:")
train_df.info()

Training dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10003 entries, 0 to 10002
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      10003 non-null  object
 1   category  10003 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


In [13]:
print("\nTest dataset information:")
test_df.info()


Test dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3080 entries, 0 to 3079
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      3080 non-null   object
 1   category  3080 non-null   object
dtypes: object(2)
memory usage: 48.2+ KB


In [14]:
print("\nTraining columns:")
print(train_df.columns.tolist())


Training columns:
['text', 'category']


### Explore the intent labels

In [18]:
print("Unique training intents:", train_df["category"].nunique())
print("Unique test intents:", test_df["category"].nunique())

Unique training intents: 77
Unique test intents: 77


In [19]:
print("\nTraining intent counts:")
display(train_df["category"].value_counts().sort_values(ascending=False))


Training intent counts:


category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
                                                   ... 
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                             41
contactless_not_working                              35
Name: count, Length: 77, dtype: int64

In [20]:
print("\nSample intent names:")
print(sorted(train_df["category"].unique()))


Sample intent names:
['Refund_not_showing_up', 'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical_card

### Check whether every test intent appears in training

In [21]:
train_categories = set(train_df["category"].unique())
test_categories = set(test_df["category"].unique())

print("Intents present only in training:", train_categories - test_categories)
print("Intents present only in test:", test_categories - train_categories)

print("Training intent count:", len(train_categories))
print("Test intent count:", len(test_categories))

Intents present only in training: set()
Intents present only in test: set()
Training intent count: 77
Test intent count: 77


### Check duplicates

In [17]:
print("\nDuplicate training rows:", train_df.duplicated().sum())
print("Duplicate test rows:", test_df.duplicated().sum())

print(
    "\nDuplicate training text values:",
    train_df["text"].duplicated().sum()
)

print(
    "Duplicate test text values:",
    test_df["text"].duplicated().sum()
)


Duplicate training rows: 0
Duplicate test rows: 0

Duplicate training text values: 0
Duplicate test text values: 0


### Validate the columns and data

In [22]:
required_columns = {"text", "category"}

assert required_columns.issubset(train_df.columns), (
    f"Training data must contain {required_columns}"
)

assert required_columns.issubset(test_df.columns), (
    f"Test data must contain {required_columns}"
)

assert train_df["text"].notna().all(), "Training text contains missing values."
assert train_df["category"].notna().all(), "Training labels contain missing values."

assert test_df["text"].notna().all(), "Test text contains missing values."
assert test_df["category"].notna().all(), "Test labels contain missing values."

print("Required columns and missing-value checks passed.")

Required columns and missing-value checks passed.


### Save a small dataset report

In [23]:
dataset_report = {
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "train_columns": len(train_df.columns),
    "test_columns": len(test_df.columns),
    "train_intents": train_df["category"].nunique(),
    "test_intents": test_df["category"].nunique(),
    "train_missing_values": int(train_df.isnull().sum().sum()),
    "test_missing_values": int(test_df.isnull().sum().sum()),
    "train_duplicate_rows": int(train_df.duplicated().sum()),
    "test_duplicate_rows": int(test_df.duplicated().sum())
}

report_df = pd.DataFrame(
    list(dataset_report.items()),
    columns=["Metric", "Value"]
)

display(report_df)

report_path = PROCESSED_DATA_DIR / "phase1_dataset_report.csv"
report_df.to_csv(report_path, index=False)

print("Report saved to:", report_path)

,Metric,Value
0,train_rows,10003
1,test_rows,3080
2,train_columns,2
3,test_columns,2
4,train_intents,77
5,test_intents,77
6,train_missing_values,0
7,test_missing_values,0
8,train_duplicate_rows,0
9,test_duplicate_rows,0


Report saved to: c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\data\processed\phase1_dataset_report.csv
